In [ ]:
# =====================================================================
# 1. 필수 패키지 설치 및 환경 설정
# =====================================================================
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas matplotlib seaborn gradio easyocr

# [핵심] 업로드해 둔 dataset.zip.zip 압축 해제 (자동 폴더 생성)
!unzip -q -o /content/dataset.zip.zip -d /content/

import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import easyocr
from tqdm import tqdm

import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import gradio as gr

# 난수 시드 고정 및 디바이스(GPU) 설정
seed = 7
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🎯 현재 연산 디바이스: {device}")

# =====================================================================
# 2. 이미지 폴더에서 EasyOCR로 데이터셋 즉석 복구 및 생성
# =====================================================================
reader = easyocr.Reader(['ko', 'en'], gpu=True)
correct_folder = '/content/correct'
phishing_folder = '/content/phishing'

data_list = []

print("\n--- [1/2] 정상 데이터 OCR 추출 시작 ---")
if os.path.exists(correct_folder):
    for filename in tqdm(os.listdir(correct_folder)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(correct_folder, filename)
            try:
                result = reader.readtext(image_path, detail=0)
                data_list.append({'text': " ".join(result), 'label': 0})
            except:
                pass
else:
    print("❌ correct 폴더를 찾을 수 없습니다. 왼쪽 창에 dataset.zip.zip이 있는지 확인하세요.")

print("\n--- [2/2] 피싱 데이터 OCR 추출 시작 ---")
if os.path.exists(phishing_folder):
    for filename in tqdm(os.listdir(phishing_folder)):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(phishing_folder, filename)
            try:
                result = reader.readtext(image_path, detail=0)
                data_list.append({'text': " ".join(result), 'label': 1})
            except:
                pass
else:
    print("❌ phishing 폴더를 찾을 수 없습니다. 왼쪽 창에 dataset.zip.zip이 있는지 확인하세요.")

if not data_list:
    raise ValueError("❌ 추출된 데이터가 없습니다. 코랩 왼쪽 파일 창에 'dataset.zip.zip' 파일이 완전히 업로드되었는지 확인해 주세요!")

# 판다스 변환 및 다음에 세션이 끊겨도 쓸 수 있게 새로 CSV 자동 백업 저장
df = pd.DataFrame(data_list)
df['text'] = df['text'].fillna('')
df.to_csv('/content/phishing_dataset.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 데이터 정제 및 'phishing_dataset.csv' 복구 완료! (총 샘플 수: {len(df)}개)")

# =====================================================================
# 3. 데이터 분리 및 토큰화 파이프라인
# =====================================================================
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=seed, stratify=df['label']
)

model_name = "skt/kobert-base-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =====================================================================
# 4. KoBERT 모델 가중치 로드 및 Trainer 빌드
# =====================================================================
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("\n🚀 KoBERT 기반 피싱 분류 Fine-tuning 시작...")
trainer.train()

# =====================================================================
# 5. 모델 평가지표 검증 및 시각화
# =====================================================================
print("\n🔎 테스트 데이터셋 검증 중...")
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=1)
y_test = test_df['label'].values

print("\n=== 📝 KoBERT 상세 평가지표 ===")
print(classification_report(y_test, y_pred, target_names=['Normal(0)', 'Phishing(1)']))

history = trainer.state.log_history
train_loss = [log['loss'] for log in history if 'loss' in log]
eval_loss = [log['eval_loss'] for log in history if 'eval_loss' in log]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_loss, label='Train Loss')
if eval_loss:
    plt.plot(eval_loss, label='Validation Loss')
plt.title('KoBERT Model Loss')
plt.ylabel('Loss')
plt.xlabel('Steps / Epochs')
plt.legend(loc='upper right')

plt.subplot(1, 2, 2)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal(0)', 'Phishing(1)'],
            yticklabels=['Normal(0)', 'Phishing(1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('KoBERT Confusion Matrix')
plt.tight_layout()
plt.show()

# =====================================================================
# 6. 실시간 시연용 Gradio 웹 UI 연동 모듈
# =====================================================================
def web_predict_phishing(image):
    if image is None:
        return "⚠️ 시연할 웹사이트 스크린샷 이미지를 업로드해 주세요."

    result = reader.readtext(image, detail=0)
    extracted_text = " ".join(result)

    if not extracted_text.strip():
        return "🍏 [안전] 이미지에서 추출된 텍스트가 없어 안전한 페이지로 판단됩니다."

    inputs = tokenizer(extracted_text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    phishing_prob = probabilities[1]
    pred_class = int(phishing_prob > 0.5)

    report =  f"🔎 [이미지 스캔 문장]:\n\"{extracted_text}\"\n\n"
    report += f"📊 #-- 피싱 위험 확률 --# : {phishing_prob*100:.2f}%\n"
    report += f"📊 #-- 최종 판별 클래스 --# : {pred_class} (0:정상, 1:피싱)\n"
    report += "="*45 + "\n"

    if pred_class == 1:
        report += f"🚨 [경고] 피싱 위험도가 매우 높습니다! 해당 웹사이트 접속을 차단합니다."
    else:
        report += f"🍏 [안전] 피싱 위험 요소가 없는 깨끗한 정상 사이트입니다."

    return report

demo = gr.Interface(
    fn=web_predict_phishing,
    inputs=gr.Image(type="filepath", label="📸 웹사이트 캡처 이미지 업로드"),
    outputs=gr.Textbox(label="🚨 실시간 인공지능 분석 결과 리포트", lines=10),
    title="🛡️ KoBERT 기반 실시간 웹 피싱 차단 시스템 (Anti-Phishing System)",
    description="웹사이트 스크린샷 이미지를 올리면, EasyOCR로 텍스트를 추출한 뒤 KoBERT 딥러닝 모델이 실시간으로 피싱 여부를 판별합니다. (최종 모델 정확도: 92%)",
    theme="soft"
)

demo.launch(share=True, inline=True)